# The trial was run in the north. The rollout is national.

Nothing is wrong with the trial. It was randomized, it was powered, the effect is identified,
the interval is tight. The question is whether its number is the number for a population it
did not sample — and "the effect is the effect" is not an answer, because the effect is an
average over a mix of people, and the mix changed.

A selection diagram is the causal graph plus S-nodes on the mechanisms that differ between
the study population and the target. `CausalGraph.selection` lists those nodes;
`selection_diagram` makes the S-nodes explicit. Transportability is then d-separation
(Bareinboim & Pearl 2014): the effect transports *directly* if $Y \perp S \mid X$ in
$G_{\overline{X}}$, and via the transport formula
$P^*(y \mid do(x)) = \sum_z P(y \mid do(x), z)\,P^*(z)$ if some $Z$ is S-admissible.

The point is not that transporting is hard. It is that whether it is licensed is a structural
question with an answer, and the answer is sometimes no.

In [ ]:
import numpy as np

from axiom.identify import (
    CausalGraph, TransportVerdict, directly_transportable, identify, minimal_s_admissible_sets,
    ols, s_admissible, s_admissible_sets, selection_diagram, transport_verdict,
    trivially_transportable,
)
from axiom.sim import transport_pair
from axiom.viz import causal_graph

from axiom.display import enable, table

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import caption, compare, density

enable();  # every axiom result renders itself from here on

In [ ]:
g = CausalGraph.from_edges("Z -> X, Z -> Y, X -> Y", selection=["Z"])
sd = selection_diagram(g)
print(sd.nodes, "| unmeasured:", sd.unmeasured)
print("direct:", directly_transportable(g, "X", "Y"))
print("S-admissible {Z}:", s_admissible(g, "X", "Y", ["Z"]), "| {}:", s_admissible(g, "X", "Y", []))
print(s_admissible_sets(g, "X", "Y"), minimal_s_admissible_sets(g, "X", "Y"))
print("trivially (from target data alone):", trivially_transportable(g, "X", "Y"))

In [ ]:
causal_graph(sd)

The square-ish extra node is the whole content of the claim: *this* mechanism differs between
the populations, and no others do. Everything below is derived from that one drawn assertion.

## Verdicts

`transport_verdict` searches for a route; pass `given=` to check a specific set — the empty
set is "read the source effect as the target's, unadjusted", which the graph here blocks.
When no sufficient condition the module implements holds, the verdict is `unsupported`
(the complete sID algorithm is out of scope for 1.0), never a false `blocked`.

In [ ]:
unadjusted: TransportVerdict = transport_verdict(g, "X", "Y", given=())
print(unadjusted.verdict.status, "|", unadjusted.verdict.reason)
adjusted = transport_verdict(g, "X", "Y")
print(adjusted.verdict.status, adjusted.route, adjusted.s_admissible_set, "|", adjusted.formula)
print([f"{a.name}:{a.state}" for a in adjusted.verdict.assumptions])

In [ ]:
cases = {
    "S only on X": CausalGraph.from_edges("Z -> X, Z -> Y, X -> Y", selection=["X"]),
    "S on Y, back-door in target": CausalGraph.from_edges("Z -> X, Z -> Y, X -> Y", selection=["Y"]),
    "S on mediator": CausalGraph.from_edges("X -> M, M -> Y", selection=["M"]),
    "no S-nodes": CausalGraph.from_edges("X -> Y"),
}
rows = []
for label, graph in cases.items():
    tv = transport_verdict(graph, "X", "Y")
    rows.append([label, tv.verdict.status, tv.route, str(tv.formula)])
table(rows, headers=("case", "status", "route", "formula"))

Read the first two rows together. If only the *assignment* of treatment differs between
populations — a different targeting rule, a different price — the effect transports
untouched. If the *outcome mechanism* differs, it does not, and no amount of reweighting
fixes it. Those two cases are indistinguishable in the data and are distinguished here by the
position of one node.

In [ ]:
v = identify(g, "X", "Y")
print(v.route, v.status, "| transport:", v.transport.verdict.status if v.transport else None)

## Recovery on a pair of worlds

`transport_pair()` gives two linear worlds differing only in the distribution of `Z`. The
slope of `X` is invariant; the *level* of the effect is not. Applying the transport formula
(fit in the source, standardize over the target's `Z`) recovers the target truth; reading the
source number as the target's misses by exactly the amount the formula predicts.

In [ ]:
source, target = transport_pair()
s = source.observed(source.simulate(40_000, seed=0))
t = target.observed(target.simulate(40_000, seed=1))
x0 = 1.0
truth = target.interventional_mean("Y", intervene={"X": x0})

design = np.column_stack([np.ones(len(s)), s["X"], s["Z"]])
a, b_x, b_z = np.linalg.lstsq(design, s["Y"].to_numpy(), rcond=None)[0]
transported = a + b_x * x0 + b_z * t["Z"].mean()
untransported = a + b_x * x0 + b_z * s["Z"].mean()
print(f"target truth E*[Y|do(X=1)] = {truth:.3f}")
print(f"transported {transported:.3f} | untransported {untransported:.3f} | predicted gap {b_z * (t['Z'].mean() - s['Z'].mean()):.3f}")
print("slope estimate in the source:", ols(s, "Y", "X", covariates=["Z"]).estimate.__round__(3))

In [ ]:
fig = density(
    {"source population": s["Z"].to_numpy(), "target population": t["Z"].to_numpy()},
    title="The one thing that differs",
    subtitle="the distribution of Z in the two worlds — the mechanism the S-node marks",
    x_title="Z",
)
caption(fig, "Nothing about the causal mechanism changed. The populations are made of "
             "different people, which is enough to move the answer.")

In [ ]:
fig = compare(
    ["source number, read as the target's", "transport formula applied", "target truth"],
    [untransported, transported, truth],
    highlight="target truth",
    value_fmt="{:.2f}",
    title="What the formula is worth",
    subtitle="E*[Y | do(X=1)] in the target population, three ways of getting there",
    x_title="expected outcome under the intervention",
)
caption(fig, "The gap between the first two bars is predicted in advance by the fitted "
             "coefficient and the difference in means — it is not a discovery made after the "
             "rollout underperformed.")

## What this bought you

A named, checkable answer to "does this result apply here?", the formula that repairs it when
one exists, and an explicit refusal when none does — instead of a footnote about
generalizability that everyone skims.

`nbs/estimands/02-transfer-plans.ipynb` turns this verdict into a ledger of assumptions that
travels with the number, and `nbs/calibrate/04-transfer-and-ledger.ipynb` is where borrowed
evidence pays for its transfer.